In [ ]:
# --- 修复部分开始 ---
# 1. 强制安装 zstd 解压工具 (修复报错的关键)
!sudo apt-get update && sudo apt-get install -y zstd
# --- 修复部分结束 ---

# 2. 安装 Ollama
!curl -fsSL https://ollama.com/install.sh | sh
!pip install aiohttp gradio
# 新增 RAG 依赖
!pip install aiohttp gradio langchain langchain-community faiss-cpu sentence-transformers

In [10]:
import os
import time
import subprocess
import threading

# 3. 在后台启动 Ollama 服务
def start_ollama():
    os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
    os.environ['OLLAMA_ORIGINS'] = '*'
    subprocess.Popen(["ollama", "serve"])

ollama_thread = threading.Thread(target=start_ollama)
ollama_thread.start()

# 等待几秒钟让服务启动
print("正在启动 Ollama 服务，请稍候...")
time.sleep(10)

# 4. 下载模型 (如果之前没下载成功)
print("正在下载 Llama 3.1 模型...")
!ollama run llama3.1 "你好"

正在启动 Ollama 服务，请稍候...
正在下载 Llama 3.1 模型...
你好！我是中文聊天机器人。能为你提供什么帮助吗？



In [ ]:
#-----------------新增RAG部分--------------------
import os
import glob
from pathlib import Path
from google.colab import drive

# ==========================================
# 🔧 自动兼容导入 (核心修复部分)
# ==========================================

# 1. 修复 Document 导入
try:
    from langchain_core.documents import Document
except ImportError:
    try:
        from langchain.schema import Document
    except ImportError:
        from langchain.docstore.document import Document

# 2. 修复 TextSplitter 导入
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

# 3. 修复 Embeddings 和 VectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# ==========================================
# RAG 数据处理逻辑
# ==========================================

# A. 挂载 Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
    print("Google Drive 挂载成功")
else:
    print("Google Drive 已经在运行中")

# B. 设置数据路径
# 确保路径指向你的 RAG 文件夹 (请在 Drive 确认路径无误)
kb_path = "/content/drive/MyDrive/Colab Notebooks/RAG/data/knowledge_base"

# C. 自定义加载函数
def load_hwu_documents(root_dir):
    docs = []
    # 递归查找所有 .md 文件
    md_files = glob.glob(os.path.join(root_dir, "**/*.md"), recursive=True)

    print(f"正在扫描路径: {root_dir}")

    if not md_files:
        print(f"未找到文件！请检查路径是否正确。")
        print(f"当前尝试读取的路径是: {root_dir}")
        return []

    print(f"发现 {len(md_files)} 个 Markdown 文件，开始加载...")

    for file_path in md_files:
        try:
            path = Path(file_path)
            text = path.read_text(encoding="utf-8")

            # 解析头部元数据
            lines = text.split("\n", 2)
            if len(lines) < 3: continue

            source = lines[0].replace("Source:", "").strip()
            topic = lines[1].replace("Topic:", "").strip()
            content = lines[2].strip()

            doc = Document(
                page_content=content,
                metadata={"source": source, "topic": topic, "filename": path.name}
            )
            docs.append(doc)
        except Exception as e:
            print(f"⚠️ 加载失败 {file_path}: {e}")
    return docs

# D. 初始化与构建
def initialize_rag_from_drive():
    if not os.path.exists(kb_path):
        print(f"错误：找不到路径 {kb_path}")
        print("请确认 'RAG' 文件夹是否放在 'Colab Notebooks' 里")
        return None

    # 加载
    documents = load_hwu_documents(kb_path)

    if not documents:
        return None

    print(f"成功加载 {len(documents)} 篇文档")

    # 切分
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
    chunks = text_splitter.split_documents(documents)
    print(f"切分为 {len(chunks)} 个片段")

    # 向量化
    print("正在构建向量索引 (Loading Embedding Model)...")
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    vector_db = FAISS.from_documents(chunks, embeddings)
    print("RAG 向量库构建完成！")
    return vector_db

# 执行
vector_db = initialize_rag_from_drive()
#------------RAG部分结束-----------------------

In [ ]:
import gradio as gr
import json
import requests

# --- 4. 启动 Chatbot 界面 (已修复 AttributeError) ---

def chat_response(message, history):
    url = "http://localhost:11434/api/chat"

    # --- RAG 检索 ---
    context_text = ""
    references = []

    # 检查向量库是否存在
    if 'vector_db' in globals() and vector_db:
        try:
            # 检索 Top 3
            retriever = vector_db.as_retriever(search_kwargs={"k": 3})

            # 【修复点】: 使用 invoke 替代旧的 get_relevant_documents
            docs = retriever.invoke(message)

            context_parts = []
            for i, doc in enumerate(docs):
                src = doc.metadata.get('source', 'Unknown URL')
                topic = doc.metadata.get('topic', 'General')
                if src not in references: references.append(src)
                context_parts.append(f"[Document {i+1} - Topic: {topic}]\n{doc.page_content}")

            context_text = "\n\n".join(context_parts)
        except Exception as e:
            print(f"RAG 检索警告: {e}")
            # 如果检索失败，不中断程序，只是 context 为空
            context_text = ""

    # --- Prompt 逻辑 ---
    system_instruction = f"""
    You are the AI Student Services Assistant for Heriot-Watt University (HWU).

    Here is the retrieved knowledge from the university database:
    [CONTEXT START]
    {context_text}
    [CONTEXT END]

    You must strictly follow these logical steps:

    **Step 1: Check Topic Relevance**
    Is the user's question about Student Services, Campus Life, or University Administration?
    - IF NO (e.g., football, coding, math, general chat):
      REPLY EXACTLY: "I apologize, I can only answer questions regarding Student Services at Heriot-Watt University."
      (STOP HERE).

    **Step 2: Check University Scope**
    Is the question specifically about Heriot-Watt University?
    - IF NO (e.g., Edinburgh University, general UK visa rules not specific to HWU):
      REPLY EXACTLY: "I apologize, I can only answer questions regarding Heriot-Watt University."
      (STOP HERE).

    **Step 3: Check Context Availability**
    Is the answer in the [CONTEXT] above?
    - IF YES: Answer politely using ONLY the context.
    - IF NO: REPLY EXACTLY: "I apologize, I do not have this specific information at the moment."

    **Output Guidelines:**
    - If the user asks in Chinese, translate the standard replies into natural Chinese.
    """

    messages_payload = [{"role": "system", "content": system_instruction}]
    for user_msg, ai_msg in history:
        messages_payload.append({"role": "user", "content": user_msg})
        messages_payload.append({"role": "assistant", "content": ai_msg})
    messages_payload.append({"role": "user", "content": message})

    payload = {"model": "llama3.1", "messages": messages_payload, "stream": True}

    # --- 请求与生成 ---
    full_response = ""
    try:
        # 设置 timeout 防止死锁
        response = requests.post(url, json=payload, stream=True, timeout=60)
        response.raise_for_status()

        for line in response.iter_lines():
            if line:
                decoded_line = line.decode('utf-8')
                json_line = json.loads(decoded_line)
                if 'message' in json_line:
                    content = json_line['message'].get('content', '')
                    full_response += content
                    yield full_response

        # 追加引用
        if references:
            ref_text = "\n\n---\n**References:**\n"
            for idx, ref in enumerate(references):
                ref_text += f"{idx+1}. {ref}\n"
            full_response += ref_text
            yield full_response

    except requests.exceptions.ConnectionError:
        yield "错误：无法连接到 Ollama 服务。请确认您已运行‘启动 Ollama 服务’的代码块。"
    except Exception as e:
        yield f"Error: {str(e)}"

# 启动 Gradio
demo = gr.ChatInterface(
    chat_response,
    title="HWU Student Services Bot (RAG)",
    description="Ask about enrollment, accommodation, or finance.",
    examples=["How do I register?", "Accommodation costs?", "Where is the library?"],
)

demo.launch(share=True, debug=True)